# Biopython으로 폐암 데이터 받아서 분석해보기 
## 개요
암... 현대인에게는 여전히 풀어야 할 숙제입니다. 물론 조기에 진단을 받으면 살 수도 있지만, 때가 늦어지면... 아... 의사쌤이 뭔가 심각한 얼굴로 부르기 시작하는데... 

그거 아십니까? 암은 다양한 요인들에 의해 발생하고 그 중 하나가 유전자 변이입니다. 옹코진(Oncogene-종양 유전자)이라는 놈이 있는데 이놈은 변이가 돼서 미쳐 날뛰는 순간 암이 되는거고, 암 억제 유전자는 변이가 터져서 제 기능을 못 하게 되면 암으로 발전하게 됩니다. 물론 우리 몸은 신호체계로 돌아가는거라 어떻게든 막고 막고 막는 경로가 존재는 하겠지만, 그 경로가 다 뻑나면... 아... 

그러니까 여러분은 담배를 멀리하시고 건강한 삶을 사시는 게 좋습니다. 돌연변이원 중 하나가 담배임. 

## 결론
**담배 끊으십쇼.**

## 프로젝트 정보
- 인원: 1인(개인 프로젝트)
- 버전: 3.10(TF_base)
- 설치할 것들: Biopython, lifelines(생존곡선 잘그려줌), GEOparse(NCBI GEO에 접근할 때 필요)
- 데이터 리소스: NCBI(Entrez로 갖고올 예정)

In [ ]:
# 모듈
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import KaplanMeierFitter # 이친구가 생존곡선을 잘 그려요 아무튼 그럼 
import GEOparse # NCBI GEO에 접근할 때 필요함 

from Bio import Entrez # NCBI 창고털이 드가자 

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Nanumbarunpen'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# NCBI 창고를 털려면 이메일이 필요함 
Entrez.email = "blackholekun@gmail.com" # 이메일 

In [ ]:
# 창고는 열려있다! 데이터를 털어라! 
print("--- GSE30219 데이터 다운로드 중... ---")
gse_mut = GEOparse.get_GEO(geo="GSE30219", destdir="./") # 진짜 창고 터는중 
df_mut = gse_mut.phenotype_data

In [ ]:
# 정보 확인
df_mut.info() # 거 정보좀 봅시다. 
df_mut.isna().sum() # 아... 널이 있었어... 
df_mut.head()

In [ ]:
# 필요한 것만 쏙 빼오기 
df_target = df_mut[[
    'characteristics_ch1.3.histology', 
    'characteristics_ch1.7.follow-up time (months)', 
    'characteristics_ch1.8.status'
]].copy()
# Histology: 암종
# Months: 생존기간
# Status: 생존 상태
df_target.columns = ['Histology', 'Months', 'Status'] # 컬럼 이름 바꿀거임 

# 선가공 
df_target['Months'] = pd.to_numeric(df_target['Months'], errors='coerce')
# 돌아가심->1, 살아있음->0
df_target['Event'] = df_target['Status'].apply(lambda x: 1 if str(x).upper() == 'DEAD' else 0)

# 암종 크게 분류 (NSCLC vs SCLC)
# NSCLC: 비소세포성 폐암
# SCLC: 소세포성 폐암
def classify_lung_cancer(x):
    if x in ['ADC', 'SQC', 'LCC', 'LCNE', 'BAS']: return 'NSCLC (Non-Small Cell)'
    elif x == 'SCC': return 'SCLC (Small Cell)'
    elif x == 'NTL': return 'Normal/Control'
    else: return 'Other'

df_target['Group'] = df_target['Histology'].apply(classify_lung_cancer)

# 생존 분석 시각화
kmf = KaplanMeierFitter()
plt.figure(figsize=(10, 6))

for name, grouped_df in df_target.groupby('Group'):
    if name == 'Normal/Control': continue 
    valid_data = grouped_df.dropna(subset=['Months'])
    if len(valid_data) > 0:
        kmf.fit(valid_data['Months'], valid_data['Event'], label=name)
        kmf.plot_survival_function()

plt.title("Survival Rate: NSCLC vs SCLC (GSE30219)")
plt.xlabel("Months")
plt.ylabel("Survival Probability")
plt.grid(True)
plt.show()

# 데이터 확인용 (인원수)
print(df_target['Group'].value_counts())

In [ ]:
# SCLC, NSCLC간 생존율 차이 
survival_clc = df_target.groupby('Group').mean('Month').sort_values(by = 'Months', ascending = False)

plt.figure(figsize = (10, 6))
plt.title('SCLC vs NSCLC Survival Rate')
plt.xlabel('Group')
plt.ylabel('Months')
sns.barplot(survival_clc, x = 'Group', y = 'Months', hue = 'Group', palette = 'coolwarm')
plt.show()

print("--- [최종 분석] 주요 암종별 평균 생존 기간 ---")
print(df_target.groupby('Group')['Months'].mean().sort_values(ascending=False))

In [ ]:
# 암종별 생존곡선 그리기
# 위에 그거...같은데? 
df_study = df_mut[[
    'characteristics_ch1.3.histology',
    'characteristics_ch1.7.follow-up time (months)',
    'characteristics_ch1.8.status'
]].copy()
df_study.columns = ['Histology', 'Months', 'Status']

# 전처리: 숫자 변환 및 사망 이벤트 정의
df_study['Months'] = pd.to_numeric(df_study['Months'], errors='coerce')
# 위랑 같음. 사망->1, 살아있음->0
df_study['Event'] = df_study['Status'].apply(lambda x: 1 if str(x).upper() == 'DEAD' else 0)

# 암종별 비교 
# ADC: 선암
# SQC: 편평상피세포암
# NTL: 정상 폐
# BAS: 기저세포양 편평세포암
# LCC: 대세포암
# LCNE: 대세포 신경내분비암
# SCC: 소세포암
# CARCI: 유암종
# Other: 기타
plt.figure(figsize=(12, 8))
ax = plt.subplot(111)
kmf = KaplanMeierFitter()

# 정상 제외하고 분석 
groups = df_study[df_study['Histology'] != 'NTL']['Histology'].unique()

for name in groups:
    group_data = df_study[df_study['Histology'] == name].dropna(subset=['Months'])
    if len(group_data) > 5: # 샘플 수가 너무 적은 그룹 제외
        kmf.fit(group_data['Months'], group_data['Event'], label=name)
        kmf.plot_survival_function(ax=ax)

plt.title("Lung Cancer Survival by Histology (GSE30219)", fontsize=15)
plt.xlabel("Months")
plt.ylabel("Survival Probability")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 인원수 및 사망자 확인
print("--- 그룹별 현황 ---")
print(df_study.groupby('Histology')['Event'].agg(['count', 'sum']).rename(columns={'sum': 'deaths'}))

In [ ]:
# 암종별 평균 생존률(막대그래애프)
survival_stats = df_study.groupby('Histology')['Months'].mean().sort_values(ascending=False)

sns.barplot(x=survival_stats.index, y=survival_stats.values, hue = survival_stats.index, palette='coolwarm')
plt.title('Average Survival Months by Histology')
plt.xlabel('Histology')
plt.ylabel('Average Survival Months')
plt.tight_layout()
plt.show()

print("--- [최종 분석] 주요 암종별 평균 생존 기간 ---")
print(df_study.groupby('Histology')['Months'].mean().sort_values(ascending=False))